# Python UAI324 bez řešení

# Cvičení bez řešení (pro studenty)

Následuje 12 zadání v rostoucí obtížnosti. Ke každému je hint a kritérium úspěchu.

### Cvičení 1 – Comprehension *(lehké)*
Pomocí dict comprehension vytvoř slovník `{slovo: len(slovo)}` pro seznam slov, ale **jen** pro slova delší než 3 znaky a převeď je na malá písmena.
> **Hint:** kombinuj `if` a metodu `.lower()` uvnitř comprehension.

### Cvičení 2 – Generátor *(lehké)*
Napiš generátor `chunks(iterable, n)`, který postupně vrací seznamy o velikosti `n` (poslední může být kratší).
> **Hint:** `iter()` + `islice`, nebo klasický cyklus se slice.

### Cvičení 3 – Dekorátor *(střední)*
Napiš dekorátor `@loguj_soubor("log.txt")`, který při každém volání funkce připíše do souboru čas, jméno funkce a argumenty.
> **Hint:** vrací dekorátor (3 úrovně), `datetime.now()`, otevírej v režimu `"a"`.

### Cvičení 4 – Context manager *(střední)*
Vytvoř context manager `docasna_změna(obj, attr, hodnota)`, který atribut nastaví a po opuštění bloku vrátí původní hodnotu – i když nastane výjimka.
> **Hint:** `@contextmanager` + `try/finally`.

### Cvičení 5 – OOP *(střední)*
Implementuj třídu `Zlomek` s operátory `+`, `-`, `*`, `/`, `==` a automatickým krácením pomocí `math.gcd`. Přidej `__repr__`.
> **Hint:** při sčítání použij společný jmenovatel; krácení v `__init__` nebo v `__post_init__`.

### Cvičení 6 – Výjimky *(střední)*
Napiš funkci `bezpecne_deleni(a, b)`, která vrátí `None` při dělení nulou, ale **vyhodí** `TypeError` s vlastní zprávou, pokud vstupy nejsou čísla.
> **Hint:** dva `except` bloky, `isinstance`, vlastní zpráva.

### Cvičení 7 – Regex *(střední)*
Z textu vytáhni všechna česká telefonní čísla ve formátu `+420 XXX XXX XXX` nebo `XXX XXX XXX` a normalizuj je na tvar `+420XXXXXXXXX`.
> **Hint:** `re.findall` s volitelnou skupinou `(?:\+420\s?)?`, pak `.replace(" ", "")`.

### Cvičení 8 – Kolekce *(střední)*
Ze seznamu dvojic `(student, předmět)` vytvoř slovník `{student: set(předmětů)}`. Poté najdi studenty, kteří mají **průnik** alespoň 2 předmětů s jiným studentem.
> **Hint:** `defaultdict(set)`, `itertools.combinations`, operace `&` na setech.

### Cvičení 9 – Asyncio *(těžší)*
Napiš `async` funkci `stahni_vse(urls, limit=5)`, která stáhne maximálně 5 úloh souběžně pomocí `asyncio.Semaphore` (simuluj `asyncio.sleep`). Vrať výsledky v pořadí vstupu.
> **Hint:** `asyncio.Semaphore(limit)` + `async with sem:` uvnitř workeru, `gather` zachová pořadí.

### Cvičení 10 – pytest *(těžší)*
Pro funkci `prevod_meny(castka, kurz)` napiš testy: (a) parametrizované platné případy, (b) test na zápornou částku, (c) test na nulový kurz. Použij fixture pro kurz.
> **Hint:** `@pytest.mark.parametrize`, `pytest.raises`, `round` na 2 desetinná místa.

### Cvičení 11 – sqlite3 *(těžší)*
Rozšiř příklad 25 o příkaz `vypujc(conn, kniha_id, jmeno)` a tabulku `vypujcky` s cizím klíčem. Zajisti, že knihu nelze vypůjčit dvakrát současně.
> **Hint:** `FOREIGN KEY`, `UNIQUE(kniha_id) WHERE vraceno IS NULL` (partial index), nebo kontrola před insertem v transakci.

### Cvičení 12 – Kombinace *(těžké)*
Postav CLI nástroj `analyza.py`, který: přijme cestu k textovému souboru přes `argparse`, spočítá top 5 slov (`Counter`), zapíše výsledek do JSON, změří dobu běhu dekorátorem a ošetří chybějící soubor vlastní výjimkou. Napiš k němu alespoň 3 pytest testy.
> **Hint:** kombinace příkladů 5, 13, 18, 23, 24.

---

## Metodické poznámky pro výuku

| Téma | Doporučený přístup |
|---|---|
| **asyncio** | Nejdřív ukaž `time.sleep` vs `asyncio.sleep` na měření času – studenti ihned vidí rozdíl. |
| **pytest** | Nech je nejdřív psát testy "ručně" (`assert`), pak ukaž, co vše pytest zjednoduší. |
| **argparse** | Začni jedním argumentem, postupně přidávej sub-příkazy – ne rovnou celý příklad 24. |
| **sqlite3** | Zdůrazni parametrizaci (`?`) – ukaž demo SQL injection s f-stringem. |
| **deskriptory** | Přemosť od `@property` ("to už znáte") k deskriptoru ("takhle to funguje uvnitř"). |
| **dekorátory** | Trénuj pattern "dekorátor bez parametru → s parametrem → třídní" postupně. |

**Hodnocení cvičení:** doporučuji bodovat (a) funkčnost, (b) čistotu kódu, (c) ošetření chyb, (d) testy – každé 25 %.

Chceš k vybraným cvičením připravit **vzorová řešení** (např. 1–6 pro rychlou kontrolu, 7–12 nechat na studentech), nebo připravit **startovací šablony** souborů (`test_*.py`, `analyza.py` s prázdnými funkcemi)?

In [ ]:
# Rozšíření sady – pokročilejší témata + cvičení bez řešení

Níže navazuji na předchozí sadu. Přidávám **6 nových řešených příkladů** (asyncio, pytest, argparse, databáze, deskriptory, třídní dekorátory) a na konci **12 cvičení bez řešení** pro studenty.

---

## 12. Asyncio

### Příklad 21: Paralelní HTTP-like úlohy s `asyncio.gather`

**Zadání:** Simuluj tři nezávislé "požadavky" trvající různě dlouho a spusť je paralelně.

```python
import asyncio
import time

async def stahni(nazev, sekundy):
    print(f"→ start {nazev}")
    await asyncio.sleep(sekundy)          # simulace I/O
    print(f"← hotovo {nazev}")
    return f"{nazev}: OK"

async def main():
    start = time.perf_counter()
    vysledky = await asyncio.gather(
        stahni("A", 1),
        stahni("B", 2),
        stahni("C", 1),
    )
    print(vysledky)
    print(f"Celkem: {time.perf_counter() - start:.2f} s")  # ~2 s, ne 4 s

asyncio.run(main())
```

**Vysvětlení:** `asyncio.gather` spustí korutiny souběžně. Celkový čas = maximum, ne součet. `await asyncio.sleep` uvolní event loop – klíčový rozdíl od `time.sleep`.

### Příklad 22: Ochrana proti "pomalému" požadavku – `wait_for`

**Zadání:** Přidej timeout a ošetři `TimeoutError`, aby jeden pomalý požadavek nezdržel ostatní.

```python
import asyncio

async def stahni(nazev, sekundy):
    await asyncio.sleep(sekundy)
    return nazev

async def s_timeoutem(nazev, sekundy, limit):
    try:
        return await asyncio.wait_for(stahni(nazev, sekundy), timeout=limit)
    except asyncio.TimeoutError:
        return f"{nazev}: TIMEOUT"

async def main():
    vysledky = await asyncio.gather(
        s_timeoutem("rychly", 0.5, 1.0),
        s_timeoutem("pomaly", 3.0, 1.0),
        s_timeoutem("stredni", 1.0, 2.0),
    )
    print(vysledky)
    # ['rychly', 'pomaly: TIMEOUT', 'stredni']

asyncio.run(main())
```

**Vysvětlení:** `wait_for` zruší korutinu po limitu a vyhodí `TimeoutError`. `gather` s `return_exceptions=True` by šel použít, pokud chceš sbírat výjimky bez přerušení ostatních.

---

## 13. Testování s pytest

### Příklad 23: Parametrizované testy + fixtures

**Zadání:** Otestuj funkci pro výpočet DPH s různými sazbami a společnou fixture.

```python
# soubor: test_dph.py
import pytest

def vypocitej_dph(castka, sazba):
    if castka < 0:
        raise ValueError("Záporná částka")
    return round(castka * (1 + sazba / 100), 2)

@pytest.fixture
def zakladni_castka():
    return 1000

@pytest.mark.parametrize("sazba, ocekavano", [
    (21, 1210.00),
    (12, 1120.00),
    (0,  1000.00),
])
def test_dph(zakladni_castka, sazba, ocekavano):
    assert vypocitej_dph(zakladni_castka, sazba) == ocekavano

def test_zaporna_castka():
    with pytest.raises(ValueError, match="Záporná"):
        vypocitej_dph(-100, 21)
```

**Spuštění:**

```bash
pytest test_dph.py -v
```

**Vysvětlení:** `@pytest.mark.parametrize` vygeneruje N testů z jednoho – skvělý poměr cena/výkon. Fixture `zakladni_castka` se injektuje podle jména parametru. `pytest.raises(..., match=...)` ověří i text výjimky.

---

## 14. Argumenty příkazové řádky

### Příklad 24: CLI nástroj s `argparse` a sub-příkazy

**Zadání:** Vytvoř nástroj `sprava.py` s příkazy `pridej` a `seznam` pracujícími s JSON souborem.

```python
# soubor: sprava.py
import argparse
import json
from pathlib import Path

DB = Path("ukoly.json")

def nacti():
    return json.loads(DB.read_text(encoding="utf-8")) if DB.exists() else []

def uloz(data):
    DB.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def cmd_pridej(args):
    data = nacti()
    data.append({"text": args.text, "hotovo": False})
    uloz(data)
    print(f"Přidáno: {args.text}")

def cmd_seznam(args):
    for i, u in enumerate(nacti(), 1):
        stav = "✓" if u["hotovo"] else " "
        print(f"{i}. [{stav}] {u['text']}")

def main():
    parser = argparse.ArgumentParser(description="Správa úkolů")
    sub = parser.add_subparsers(dest="prikaz", required=True)

    p_add = sub.add_parser("pridej", help="Přidá úkol")
    p_add.add_argument("text")
    p_add.set_defaults(func=cmd_pridej)

    p_ls = sub.add_parser("seznam", help="Vypíše úkoly")
    p_ls.set_defaults(func=cmd_seznam)

    args = parser.parse_args()
    args.func(args)

if __name__ == "__main__":
    main()
```

**Použití:**

```bash
python sprava.py pridej "Naučit se asyncio"
python sprava.py seznam
```

**Vysvětlení:** `set_defaults(func=...)` je čistý pattern pro dispatch – každý sub-příkaz má vlastní handler. `required=True` u subparserů vynutí zadání příkazu.

---

## 15. Práce s databází (sqlite3)

### Příklad 25: CRUD operace s parametry a transakcí

**Zadání:** Vytvoř jednoduchou evidenci knih s ošetřením duplicit a transakcí.

```python
import sqlite3

def init_db(conn):
    conn.execute("""
        CREATE TABLE IF NOT EXISTS knihy (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            nazev TEXT NOT NULL UNIQUE,
            autor TEXT NOT NULL,
            rok INTEGER
        )
    """)
    conn.commit()

def pridej_knihu(conn, nazev, autor, rok):
    try:
        with conn:  # transakce – commit při úspěchu, rollback při výjimce
            conn.execute(
                "INSERT INTO knihy (nazev, autor, rok) VALUES (?, ?, ?)",
                (nazev, autor, rok),
            )
        return True
    except sqlite3.IntegrityError:
        print(f"Kniha '{nazev}' už existuje")
        return False

def vsechny(conn):
    return conn.execute("SELECT nazev, autor, rok FROM knihy ORDER BY rok").fetchall()

conn = sqlite3.connect(":memory:")
init_db(conn)
pridej_knihu(conn, "Krysař", "Viktor Dyk", 1915)
pridej_knihu(conn, "R.U.R.", "Karel Čapek", 1920)
pridej_knihu(conn, "R.U.R.", "Karel Čapek", 1920)  # duplicita

for radek in vsechny(conn):
    print(radek)
```

**Vysvětlení:** Vždy používej **parametrizované dotazy** (`?`) – chrání proti SQL injection. `with conn:` = transakce (ne uzavření spojení!). `UNIQUE` constraint vyhodí `IntegrityError`, kterou elegantně odchytíš.

---

## 16. Deskriptory

### Příklad 26: Validující deskriptor pro atribut

**Zadání:** Vytvoř deskriptor `KladneCislo`, který povolí jen kladná čísla.

```python
class KladneCislo:
    def __set_name__(self, owner, name):
        self.jmeno = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__.get(self.jmeno)

    def __set__(self, obj, hodnota):
        if not isinstance(hodnota, (int, float)) or hodnota <= 0:
            raise ValueError(f"{self.jmeno} musí být kladné číslo")
        obj.__dict__[self.jmeno] = hodnota

class Produkt:
    cena = KladneCislo()
    mnozstvi = KladneCislo()

    def __init__(self, nazev, cena, mnozstvi):
        self.nazev = nazev
        self.cena = cena
        self.mnozstvi = mnozstvi

p = Produkt("Notebook", 25000, 3)
print(p.cena, p.mnozstvi)   # 25000 3

try:
    p.cena = -5
except ValueError as e:
    print(e)                # cena musí být kladné číslo
```

**Vysvětlení:** Deskriptor je třída s `__get__`/`__set__`, jejíž instance jsou **třídní atributy**. `__set_name__` automaticky zjistí jméno atributu – nemusíš ho psát ručně. Základ pro `@property`, ORM a validaci.

---

## 17. Třídní dekorátory a registrace

### Příklad 27: Registr pluginů pomocí dekorátoru

**Zadání:** Vytvoř dekorátor `@registruj`, který zařadí funkci do globálního registru podle jména.

```python
REGISTR = {}

def registruj(jmeno=None):
    def dekorator(func):
        klic = jmeno or func.__name__
        REGISTR[klic] = func
        return func
    return dekorator

@registruj()
def pozdrav():
    return "Ahoj!"

@registruj("rozluceni")
def bye():
    return "Sbohem!"

def spust(jmeno):
    if jmeno not in REGISTR:
        raise KeyError(f"Neznámý příkaz: {jmeno}")
    return REGISTR[jmeno]()

print(spust("pozdrav"))    # Ahoj!
print(spust("rozluceni"))  # Sbohem!
print(list(REGISTR))       # ['pozdrav', 'rozluceni']
```

**Vysvětlení:** Stejný pattern používají Flask (`@app.route`), pytest (`@pytest.fixture`) i argparse pluginy. Dekorátor jen "zapíše" funkci do slovníku – dispatch pak probíhá podle jména za běhu.

---

